# Final Kaggle submissions

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

train = pd.read_csv("train.csv").sort_values("id").reset_index(drop=True)
test = pd.read_csv("test.csv")

X_train = train["text"].astype(str)
y_train = train["label"]
X_test = test["text"].astype(str)

print(len(train), "training examples,", len(test), "test examples")

## 2. Train models

In [ ]:
char_model = Pipeline([
    ("hash", HashingVectorizer(
        analyzer="char", ngram_range=(3, 5), n_features=2**17,
        alternate_sign=False, norm=None
    )),
    ("tfidf", TfidfTransformer(sublinear_tf=True)),
    ("clf", LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", random_state=42))
])

word_bigram_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2), min_df=2, max_features=110000, sublinear_tf=True
    )),
    ("clf", LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", random_state=42))
])

word_trigram_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 3), min_df=2, max_features=150000, sublinear_tf=True
    )),
    ("clf", LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", random_state=42))
])

char_model.fit(X_train, y_train)
word_bigram_model.fit(X_train, y_train)
word_trigram_model.fit(X_train, y_train)
print("Models trained")

## 3. Predict

In [ ]:
p_char = char_model.predict_proba(X_test)[:, 1]
p_word12 = word_bigram_model.predict_proba(X_test)[:, 1]
p_word123 = word_trigram_model.predict_proba(X_test)[:, 1]

blank = X_test.str.contains("\n\n", regex=False).to_numpy()
double_space = X_test.str.contains("  ", regex=False).to_numpy()

# Word bigrams + character n-grams
blend12 = 0.75 * p_char + 0.25 * p_word12
p_bigram = expit(
    logit(np.clip(blend12, 1e-6, 1 - 1e-6))
    + 2 * blank - 0.5 * (~blank) * double_space
)

# Word trigrams + character n-grams
blend123 = 0.75 * p_char + 0.25 * p_word123
p_trigram = expit(
    logit(np.clip(blend123, 1e-6, 1 - 1e-6))
    + 2 * blank - 0.5 * (~blank) * double_space
)

## 4. Save CSVs

In [ ]:
pd.DataFrame({"id": test["id"], "label": p_bigram}).to_csv(
    "word25_char75_blank2_doublespace05_v3.csv", index=False
)

pd.DataFrame({"id": test["id"], "label": p_trigram}).to_csv(
    "experiment_word_trigrams.csv", index=False
)